# Deploy MCP Server to Amazon Bedrock AgentCore Runtime

This notebook deploys the `mcp_server.py` to AgentCore Runtime using AWS IAM inbound authentication.

## Install Dependencies

In [ ]:
!pip install --force-reinstall -U mcp>=1.10.0 boto3 'bedrock-agentcore<=0.1.5' bedrock-agentcore-starter-toolkit==0.1.14 --quiet

## Setup

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client("ssm", region_name=region)

tool_name = "blog_mcp_math"

print(f"Using AWS region: {region}")
print(f"Agent runtime name: {tool_name}")

## Configure AgentCore Runtime

In [ ]:
required_files = ["mcp_server.py"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    protocol="MCP",
    agent_name=tool_name,
)
print("Configuration completed")

## Launch MCP Server to AgentCore Runtime

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

## Store Agent ARN in Parameter Store

In [ ]:
ssm_client.put_parameter(
    Name="/blog_mcp_math/runtime_iam/agent_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Description="Agent ARN for blog_mcp_math MCP server",
    Overwrite=True,
)
print("Agent ARN stored in Parameter Store")
print(f"Agent ARN: {launch_result.agent_arn}")

## Test the MCP

In [ ]:
!python mcp_client_remote.py

## Cleanup (Optional)

In [ ]:
# Uncomment to clean up resources
# try:
#     ssm_client.delete_parameter(Name="/blog_mcp_math/runtime_iam/agent_arn")
#     print("Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("Parameter Store parameter not found")

# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=tool_name,
#     delete_ecr_repo=True,
# )